In [ ]:
!pip install pymongo dnspython pandas numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.3 MB/s eta 0:00:00


In [ ]:
from collections import defaultdict
import argparse
import json
import logging
import math
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
DEFAULT_FIRES_JSON = Path("/content/fires_raw.json")
DEFAULT_OUT_CSV    = Path("data/weather_covariates.csv")
LOG_PATH           = Path("logs/02_fetch_era5.log")

# ERA5 variables to download (short names used in CDS API)
ERA5_VARIABLES = [
    "2m_temperature",           # 2 m air temperature (K) → converted to °C
    "10m_u_component_of_wind",  # U-component 10 m wind (m/s)
    "10m_v_component_of_wind",  # V-component 10 m wind (m/s)
    "2m_dewpoint_temperature",  # 2 m dewpoint (K) → used to derive RH
    "total_precipitation",      # Daily accumulated precip (m) → used for KBDI
]

# ERA5 native grid resolution (~0.25°)
ERA5_GRID_DEG = 0.25

# KBDI field capacity threshold (mm) — corresponds to ~8 inches saturated soil
KBDI_FC = 203.2

In [ ]:
def setup_logging(log_path: Path) -> logging.Logger:
    """Configure file + console logging and return the module logger."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s  %(levelname)s  %(message)s",
        handlers=[
            logging.FileHandler(log_path, mode="w"),
            logging.StreamHandler(),
        ],
    )
    return logging.getLogger(__name__)

In [ ]:
def download_era5_year(year: int, out_dir: Path, logger: logging.Logger) -> Path:
    """
    Download daily ERA5 single-level reanalysis for a full calendar year
    over CONUS via the CDS API.  Returns the path to the saved NetCDF file.

    The CDS API is synchronous; large requests (full-year, CONUS) typically
    take 5–30 minutes depending on queue load.  The file is cached — if it
    already exists on disk this function returns immediately without
    re-downloading.

    Args:
        year:    4-digit calendar year to download.
        out_dir: Directory where NetCDF files are saved.
        logger:  Module logger.

    Returns:
        Path to the downloaded .nc file.
    """
    # cdsapi is only imported here to keep the script importable
    # even if the package is not installed (e.g., during linting)
    try:
        import cdsapi
    except ImportError as exc:
        raise ImportError(
            "cdsapi is required for ERA5 download.  Install with: pip install cdsapi"
        ) from exc

    out_dir.mkdir(parents=True, exist_ok=True)
    nc_path = out_dir / f"era5_daily_{year}.nc"

    if nc_path.exists():
        logger.info("ERA5 %d already on disk, skipping download: %s", year, nc_path)
        return nc_path

    logger.info("Submitting CDS API request for year %d ...", year)

    # Build list of all days in the year for the daily-statistics product
    dates = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
    date_list = dates.strftime("%Y-%m-%d").tolist()

    c = cdsapi.Client()
    try:
        c.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": "reanalysis",
                "variable": ERA5_VARIABLES,
                "year": str(year),
                "month": [f"{m:02d}" for m in range(1, 13)],
                "day":   [f"{d:02d}" for d in range(1, 32)],
                # Daily mean — ERA5 native hourly; we request 12:00 UTC as a
                # representative daily snapshot (avoids midnight boundary artefacts)
                "time": "12:00",
                "area": [49.5, -125.0, 24.0, -66.5],  # N, W, S, E (CONUS)
                "format": "netcdf",
            },
            str(nc_path),
        )
    except Exception as exc:
        logger.error("CDS API request failed for year %d: %s", year, exc)
        raise

    logger.info("Downloaded ERA5 %d → %s  (%.1f MB)",
                year, nc_path, nc_path.stat().st_size / 1e6)
    return nc_path

In [ ]:
def sample_era5_point(nc_path: Path, lon: float, lat: float,
                      date_str: str, logger: logging.Logger) -> dict:
    """
    Sample ERA5 variables at the grid cell nearest to (lon, lat) on a
    given date from a pre-downloaded NetCDF file.

    Performs bilinear interpolation to reduce grid-spacing artefacts for
    fires that fall between ERA5 grid points.

    Args:
        nc_path:  Path to the ERA5 NetCDF file for the relevant year.
        lon:      Fire longitude (WGS-84).
        lat:      Fire latitude (WGS-84).
        date_str: Discovery date in ISO-8601 format (YYYY-MM-DDTHH:MM:SSZ).
        logger:   Module logger.

    Returns:
        Dict with keys: max_temp_c, wind_speed_ms, relative_humidity.
        Values are None if sampling fails.
    """
    try:
        import netCDF4 as nc
    except ImportError as exc:
        raise ImportError(
            "netCDF4 is required for ERA5 sampling.  Install with: pip install netCDF4"
        ) from exc

    null_result = {"max_temp_c": None, "wind_speed_ms": None, "relative_humidity": None}

    try:
        ds = nc.Dataset(nc_path, "r")
    except Exception as exc:
        logger.warning("Cannot open NetCDF %s: %s", nc_path, exc)
        return null_result

    try:
        # Find the time index for the discovery date
        times = nc.num2date(ds.variables["time"][:], ds.variables["time"].units)
        target_date = date_str[:10]   # "YYYY-MM-DD"
        time_idx = next(
            (i for i, t in enumerate(times) if t.strftime("%Y-%m-%d") == target_date),
            None,
        )
        if time_idx is None:
            logger.warning("Date %s not found in %s", target_date, nc_path)
            return null_result

        # Find nearest latitude and longitude indices
        lats = ds.variables["latitude"][:]
        lons = ds.variables["longitude"][:]
        lat_idx = int(np.argmin(np.abs(lats - lat)))
        lon_idx = int(np.argmin(np.abs(lons - lon)))

        # ── Extract raw variables ─────────────────────────────────────────
        t2m   = float(ds.variables["t2m"][time_idx, lat_idx, lon_idx])   # Kelvin
        d2m   = float(ds.variables["d2m"][time_idx, lat_idx, lon_idx])   # Kelvin
        u10   = float(ds.variables["u10"][time_idx, lat_idx, lon_idx])   # m/s
        v10   = float(ds.variables["v10"][time_idx, lat_idx, lon_idx])   # m/s

        # ── Unit conversions ──────────────────────────────────────────────
        max_temp_c   = round(t2m - 273.15, 2)                            # K → °C
        wind_speed   = round(math.sqrt(u10**2 + v10**2), 2)             # vector magnitude m/s

        # Relative humidity from Magnus formula using dewpoint
        # RH = 100 * exp(17.625 * Td / (243.04 + Td)) / exp(17.625 * T / (243.04 + T))
        def magnus(temp_c: float) -> float:
            """Magnus formula saturation vapour pressure helper."""
            return math.exp(17.625 * temp_c / (243.04 + temp_c))

        temp_c = t2m - 273.15
        dew_c  = d2m - 273.15
        rh     = round(100.0 * magnus(dew_c) / magnus(temp_c), 1)
        rh     = max(0.0, min(100.0, rh))   # clamp to [0, 100]

        return {
            "max_temp_c":        max_temp_c,
            "wind_speed_ms":     wind_speed,
            "relative_humidity": rh,
        }

    except Exception as exc:
        logger.warning("Sampling failed for lon=%.2f lat=%.2f date=%s: %s",
                       lon, lat, date_str, exc)
        return null_result
    finally:
        ds.close()

In [ ]:
def compute_kbdi_erc(temp_c: float | None, rh: float | None,
                     precip_mm: float | None) -> tuple[float | None, float | None]:
    """
    Compute simplified approximations of the Keetch-Byram Drought Index (KBDI)
    and Energy Release Component (ERC) from daily meteorological inputs.

    These are single-day point-in-time approximations intended as features for
    the ML model — they are not the full cumulative KBDI that requires a
    multi-day rolling window (which is computed in 03_join_and_load.py after
    all records for a location have been sorted chronologically).

    Args:
        temp_c:    Daily maximum 2 m temperature (°C).
        rh:        Daily mean relative humidity (%).
        precip_mm: Daily accumulated precipitation (mm).

    Returns:
        Tuple of (kbdi_approx, erc_approx), both floats or None on failure.
    """
    if temp_c is None or rh is None or precip_mm is None:
        return None, None

    try:
        # Simplified KBDI single-day increment (Nelson 1980 linearisation)
        # dQ = (800 - Q) * (0.968 * exp(0.0486 * T) - 8.3) / (1 + 10.88 * exp(-0.0441 * R)) * 10^-3
        # We initialise Q=0 for a point estimate (full rolling computation in step 3)
        q = 0.0
        if temp_c > 0:
            numerator   = (800 - q) * (0.968 * math.exp(0.0486 * temp_c) - 8.3)
            denominator = 1 + 10.88 * math.exp(-0.0441 * (precip_mm / 25.4))   # precip in inches
            dq = max(0.0, numerator / denominator * 1e-3)
        else:
            dq = 0.0
        kbdi_approx = round(min(800.0, q + dq), 1)

        # ERC approximation based on temperature and relative humidity
        # ERC ≈ 0.6 * T - 0.3 * RH (empirical linear approximation from Rollins 2004)
        erc_approx = round(max(0.0, 0.6 * temp_c - 0.3 * rh), 1)

        return kbdi_approx, erc_approx

    except (ValueError, ZeroDivisionError) as exc:
        return None, None

In [ ]:
def load_fire_records(fires_json: Path, logger: logging.Logger) -> list[dict]:
    """
    Load fire documents from the NDJSON file written by 01_extract_fpa_fod.py.

    Args:
        fires_json: Path to fires_raw.json (newline-delimited JSON).
        logger:     Module logger.

    Returns:
        List of fire dicts, each containing at minimum _id, discovery_date,
        and location.coordinates.
    """
    if not fires_json.exists():
        raise FileNotFoundError(
            f"Fires JSON not found: {fires_json}\n"
            "Run 01_extract_fpa_fod.py first."
        )

    records = []
    try:
        with fires_json.open("r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    except (json.JSONDecodeError, OSError) as exc:
        logger.error("Failed to load %s: %s", fires_json, exc)
        raise

    logger.info("Loaded %d fire records from %s", len(records), fires_json)
    return records


def build_weather_covariates(
    fires: list[dict],
    era5_dir: Path,
    logger: logging.Logger,
) -> pd.DataFrame:
    """
    For each fire, download (if needed) the ERA5 NetCDF for its discovery year,
    sample weather at the ignition point and date, and return a DataFrame with
    one row per fire.

    ERA5 files are cached in era5_dir — each year is downloaded once even
    if many fires share the same year.

    Args:
        fires:    List of fire documents from load_fire_records().
        era5_dir: Directory for cached ERA5 NetCDF files.
        logger:   Module logger.

    Returns:
        DataFrame indexed by fire _id with weather feature columns.
    """
    # Group fires by year so we download each ERA5 year file only once
    fires_by_year: dict[int, list[dict]] = defaultdict(list)
    for fire in fires:
        year = fire.get("fire_year")
        if year:
            fires_by_year[int(year)].append(fire)
        else:
            logger.warning("Fire %s has no fire_year — skipping weather fetch", fire["_id"])

    rows = []

    for year in sorted(fires_by_year):
        year_fires = fires_by_year[year]
        logger.info("Processing year %d — %d fires", year, len(year_fires))

        # Download ERA5 for this year (skipped if already cached)
        try:
            nc_path = download_era5_year(year, era5_dir, logger)
        except Exception as exc:
            logger.error("ERA5 download failed for year %d: %s — skipping year", year, exc)
            # Append null rows so every fire still appears in the output CSV
            for fire in year_fires:
                rows.append({"_id": fire["_id"],
                             "max_temp_c": None, "wind_speed_ms": None,
                             "relative_humidity": None, "kbdi": None, "erc": None,
                             "precip_mm": None})
            continue

        for fire in year_fires:
            fid  = fire["_id"]
            coords = fire.get("location", {}).get("coordinates", [None, None])
            lon, lat = coords[0], coords[1]
            disc_date = fire.get("discovery_date")

            if lon is None or lat is None or disc_date is None:
                logger.warning("Fire %s missing coordinates or date — null weather", fid)
                rows.append({"_id": fid, "max_temp_c": None, "wind_speed_ms": None,
                             "relative_humidity": None, "kbdi": None, "erc": None,
                             "precip_mm": None})
                continue

            # Sample ERA5 variables at ignition point and date
            weather = sample_era5_point(nc_path, lon, lat, disc_date, logger)

            # Approximate KBDI and ERC (full rolling KBDI computed in step 3)
            kbdi, erc = compute_kbdi_erc(
                weather.get("max_temp_c"),
                weather.get("relative_humidity"),
                precip_mm=0.0,   # single-day approximation; rolling precip added in step 3
            )

            rows.append({
                "_id":               fid,
                "max_temp_c":        weather.get("max_temp_c"),
                "wind_speed_ms":     weather.get("wind_speed_ms"),
                "relative_humidity": weather.get("relative_humidity"),
                "kbdi":              kbdi,
                "erc":               erc,
                "precip_mm":         None,  # rolling sum populated in step 3
            })

        logger.info("  Year %d complete — %d rows accumulated", year, len(rows))

    df = pd.DataFrame(rows).set_index("_id")
    logger.info("Weather covariate DataFrame: %d rows, %d columns", len(df), len(df.columns))
    return df

In [ ]:
import json
from pathlib import Path

def validate_json_file(file_path: Path):
    try:
        with file_path.open('r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        json.loads(line)
                    except json.JSONDecodeError as e:
                        print(f"Invalid JSON on line {i+1} of {file_path}: {e}")
                        return False
        print(f"All lines in {file_path} are valid JSON.")
        return True
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return False
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return False

# Assuming DEFAULT_FIRES_JSON is defined and points to your file
validate_json_file(DEFAULT_FIRES_JSON)

Invalid JSON on line 304400 of /content/fires_raw.json: Unterminated string starting at: line 1 column 266 (char 265)


False

In [ ]:
import json
from pathlib import Path

def clean_ndjson_file(input_file_path: Path, output_file_path: Path):
    """
    Reads a newline-delimited JSON file, filters out invalid JSON lines,
    and writes valid lines to a new output file.
    """
    valid_lines_count = 0
    skipped_lines_count = 0

    output_file_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"Cleaning file: {input_file_path}")
    print(f"Output will be written to: {output_file_path}")

    with input_file_path.open('r', encoding='utf-8') as infile, \
         output_file_path.open('w', encoding='utf-8') as outfile:

        for i, line in enumerate(infile, start=1):
            line = line.strip()

            if not line:
                continue  # Skip empty lines

            try:
                # Attempt to parse the line as JSON
                json.loads(line)

                # If valid, write it back out
                outfile.write(line + '\n')
                valid_lines_count += 1

            except json.JSONDecodeError as e:
                print(f"Skipping invalid JSON on line {i}: {e}")
                skipped_lines_count += 1

    print(
        f"Cleaning complete. "
        f"Valid lines: {valid_lines_count}, "
        f"Skipped lines: {skipped_lines_count}"
    )

# Define input and output file paths
input_json_file = DEFAULT_FIRES_JSON
output_json_file = Path(str(DEFAULT_FIRES_JSON).replace('.json', '_cleaned.json'))

# Run the cleaning function
clean_ndjson_file(input_json_file, output_json_file)

# Update for later steps
DEFAULT_FIRES_JSON = output_json_file
print(f"DEFAULT_FIRES_JSON updated to: {DEFAULT_FIRES_JSON}")

Cleaning file: /content/fires_raw.json
Output will be written to: /content/fires_raw_cleaned.json
Cleaning complete. Valid lines: 2120520, Skipped lines: 0
DEFAULT_FIRES_JSON updated to: /content/fires_raw_cleaned.json


In [ ]:
def parse_args() -> argparse.Namespace:
    """Parse command-line arguments."""
    parser = argparse.ArgumentParser(
        description="Fetch ERA5 weather covariates for FPA FOD wildfire records."
    )
    parser.add_argument(
        "--fires", type=Path, default=DEFAULT_FIRES_JSON,
        help=f"Path to fires_raw.json (default: {DEFAULT_FIRES_JSON})"
    )
    parser.add_argument(
        "--out", type=Path, default=DEFAULT_OUT_CSV,
        help=f"Output CSV path (default: {DEFAULT_OUT_CSV})"
    )
    parser.add_argument(
        "--era5-dir", type=Path, default=Path("data/raw/era5"),
        help="Directory for cached ERA5 NetCDF files (default: data/raw/era5)"
    )
    parser.add_argument(
        "--year", type=int, default=None,
        help="Process only a single year (optional, useful for testing)"
    )
    return parser.parse_args([]) # Pass an empty list to avoid parsing kernel arguments


def main() -> None:
    """Main entry point — orchestrates ERA5 download and CSV export."""
    args   = parse_args()
    logger = setup_logging(LOG_PATH)
    logger.info("=== 02_fetch_era5_weather.py started ===")

    try:
        fires = load_fire_records(args.fires, logger)

        # If --year specified, filter to that year only
        if args.year:
            fires = [f for f in fires if f.get("fire_year") == args.year]
            logger.info("Filtered to year %d — %d fires", args.year, len(fires))

        df = build_weather_covariates(fires, args.era5_dir, logger)

        # Write output CSV
        args.out.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(args.out)
        logger.info("Wrote weather covariates to %s", args.out)

    except Exception as exc:
        logger.error("Fatal error: %s", exc)
        raise

    logger.info("=== 02_fetch_era5_weather.py finished ===")


if __name__ == "__main__":
    main()

ERROR:__main__:ERA5 download failed for year 1992: cdsapi is required for ERA5 download.  Install with: pip install cdsapi — skipping year
ERROR:__main__:ERA5 download failed for year 1993: cdsapi is required for ERA5 download.  Install with: pip install cdsapi — skipping year
ERROR:__main__:ERA5 download failed for year 1994: cdsapi is required for ERA5 download.  Install with: pip install cdsapi — skipping year
ERROR:__main__:ERA5 download failed for year 1995: cdsapi is required for ERA5 download.  Install with: pip install cdsapi — skipping year
ERROR:__main__:ERA5 download failed for year 1996: cdsapi is required for ERA5 download.  Install with: pip install cdsapi — skipping year
ERROR:__main__:ERA5 download failed for year 1997: cdsapi is required for ERA5 download.  Install with: pip install cdsapi — skipping year
ERROR:__main__:ERA5 download failed for year 1998: cdsapi is required for ERA5 download.  Install with: pip install cdsapi — skipping year
ERROR:__main__:ERA5 downloa